# Notebook 1: preparación de los datos y *data leakage* (Etapas 0 y 1)

Notebook ejecutado en **Google Colab**. Cubre:

* **Etapa 0:** configuración, montaje de Drive y estructura de carpetas del proyecto.
* **Etapa 1:** carga de `heart.csv`, tratamiento de valores inválidos, partición
  entrenamiento/prueba **antes** de cualquier escalado y demostración de fuga de datos.

Las salidas que se muestran son las de la ejecución real en Colab. El notebook 2
continúa con el modelado (Etapa 2).

## Etapa 0: configuración, Drive y estructura de carpetas

In [ ]:
import os
import json
import time
import shutil
import hashlib
import subprocess
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    BASE_DIR = Path("/content/drive/MyDrive/Heart_Failure_Prediction")
else:  # ejecución local (pruebas)
    BASE_DIR = Path(os.environ.get("HF_BASE_DIR", "Heart_Failure_Prediction")).resolve()

# ----------------------------- OPCIONES ------------------------------------
DOWNLOAD_DATA = False   # True: vuelve a descargar heart.csv aunque ya exista en Drive
FORCE_RETRAIN = False   # True: ignora TODOS los checkpoints y reentrena todo
RETRAIN_ONLY = []       # p. ej. ["SVC", "KNN"]: reentrena solo esos modelos
RANDOM_STATE = 42
CV_FOLDS = 5
DOCKER_USER = "TU_USUARIO_DOCKER"   # para k8s/deployment.yaml
# ---------------------------------------------------------------------------

SUBDIRS = ["app", "docker", "k8s", "notebooks", ".github/workflows", "tests",
           "models", "results", "figures"]
DIRS = {d: BASE_DIR / d for d in SUBDIRS}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Proyecto en:", BASE_DIR)
print("Subcarpetas:", ", ".join(SUBDIRS))

Mounted at /content/drive
Proyecto en: /content/drive/MyDrive/Heart_Failure_Prediction
Subcarpetas: app, docker, k8s, notebooks, .github/workflows, tests, models, results, figures


## Datos: `heart.csv` (Drive o descarga automática)

Si `heart.csv` ya está en la carpeta de Drive se usa tal cual. Si no está (o si
pones `DOWNLOAD_DATA = True`) se descarga y se guarda en Drive. Orden de
intentos: `kagglehub` → CLI de Kaggle (si hay un `kaggle.json` en la carpeta del
proyecto) → `CSV_URL` (opcional).

In [ ]:
%pip install -q kagglehub

In [ ]:
DATA_PATH = BASE_DIR / "heart.csv"
KAGGLE_SLUG = "fedesoriano/heart-failure-prediction"
CSV_URL = ""   # opcional: URL directa a un heart.csv, se usa si Kaggle falla


def download_heart_csv(dest):
    """Descarga heart.csv en `dest`. Devuelve el método que funcionó."""
    dest = Path(dest)

    # 1) kagglehub (dataset público)
    try:
        import kagglehub
        folder = Path(kagglehub.dataset_download(KAGGLE_SLUG))
        shutil.copy(next(folder.rglob("heart.csv")), dest)
        return "kagglehub"
    except Exception as e:
        print("  kagglehub no funcionó:", repr(e)[:150])

    # 2) CLI de Kaggle con kaggle.json guardado en la carpeta del proyecto
    cred = BASE_DIR / "kaggle.json"
    if cred.exists():
        try:
            import tempfile
            kdir = Path.home() / ".kaggle"
            kdir.mkdir(exist_ok=True)
            shutil.copy(cred, kdir / "kaggle.json")
            os.chmod(kdir / "kaggle.json", 0o600)
            tmp = Path(tempfile.mkdtemp())
            subprocess.run(["kaggle", "datasets", "download", "-d", KAGGLE_SLUG,
                            "-p", str(tmp), "--unzip"], check=True)
            shutil.copy(next(tmp.rglob("heart.csv")), dest)
            return "kaggle CLI"
        except Exception as e:
            print("  kaggle CLI no funcionó:", repr(e)[:150])

    # 3) URL directa
    if CSV_URL:
        try:
            pd.read_csv(CSV_URL).to_csv(dest, index=False)
            return "CSV_URL"
        except Exception as e:
            print("  CSV_URL no funcionó:", repr(e)[:150])

    raise FileNotFoundError(
        f"No se pudo descargar heart.csv. Súbelo manualmente a {BASE_DIR}, "
        "guarda allí tu kaggle.json o define CSV_URL."
    )


if DOWNLOAD_DATA or not DATA_PATH.exists():
    print("Descargando heart.csv ...")
    print("OK, método:", download_heart_csv(DATA_PATH))
else:
    print("Usando heart.csv existente en Drive:", DATA_PATH)

df = pd.read_csv(DATA_PATH)
DATA_HASH = hashlib.md5(DATA_PATH.read_bytes()).hexdigest()

TARGET = next((c for c in ["HeartDisease", "target"] if c in df.columns), None)
if TARGET is None:
    raise ValueError(f"No encuentro la variable objetivo. Columnas: {list(df.columns)}")
print("Filas/columnas:", df.shape, "| objetivo:", TARGET)

Usando heart.csv existente en Drive: /content/drive/MyDrive/Heart_Failure_Prediction/heart.csv
Filas/columnas: (918, 12) | objetivo: HeartDisease


## Etapa 1: exploración y preparación

En el dataset de Kaggle `Cholesterol = 0` y `RestingBP = 0` no son valores
clínicamente posibles: se convierten a `NaN` y se imputan **dentro del
`Pipeline`** (la mediana se aprende solo con entrenamiento).
La partición train/test se hace **antes** de cualquier escalado o imputación.

In [ ]:
from sklearn.model_selection import train_test_split

display(df.head())
print("\nNulos por columna:\n", df.isnull().sum().to_string())
print("\nDistribución del objetivo:\n",
      df[TARGET].value_counts(normalize=True).round(3).to_string())

for col in ["RestingBP", "Cholesterol"]:
    if col in df.columns:
        n_zero = int((df[col] == 0).sum())
        df.loc[df[col] == 0, col] = np.nan
        print(f"{col}: {n_zero} ceros convertidos a NaN")

X = df.drop(columns=TARGET)
y = df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"\nTrain: {X_train.shape} | Test: {X_test.shape}")

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0



Nulos por columna:
 Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0

Distribución del objetivo:
 HeartDisease
1    0.553
0    0.447
RestingBP: 1 ceros convertidos a NaN
Cholesterol: 172 ceros convertidos a NaN

Train: (734, 11) | Test: (184, 11)


### 1.1 Demostración de data leakage

Cuatro escenarios con un SVC (misma partición y mismo `GridSearchCV`):
escalar **antes** del split vs. `Pipeline`, con y sin una variable artificial
`leaky_feature = y + ruido` (como en el enunciado). Se guarda en
`results/leakage_demo.json` (checkpoint).

In [ ]:
from sklearn.svm import SVC
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix,
                             roc_curve, ConfusionMatrixDisplay)

CV = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
LEAK_PATH = DIRS["results"] / "leakage_demo.json"


def auc_flow(X_df, y_s, preprocess_before_split):
    """AUC en test escalando antes del split (con fuga) o con Pipeline (sin fuga)."""
    if preprocess_before_split:
        Xp = MinMaxScaler().fit_transform(
            SimpleImputer(strategy="median").fit_transform(X_df))
        Xtr, Xte, ytr, yte = train_test_split(
            Xp, y_s, test_size=0.2, stratify=y_s, random_state=RANDOM_STATE)
        g = GridSearchCV(SVC(probability=True, random_state=RANDOM_STATE),
                         {"C": [0.1, 1, 10], "gamma": [0.01, 0.1]},
                         cv=CV, scoring="roc_auc")
    else:
        Xtr, Xte, ytr, yte = train_test_split(
            X_df, y_s, test_size=0.2, stratify=y_s, random_state=RANDOM_STATE)
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                         ("scaler", MinMaxScaler()),
                         ("svc", SVC(probability=True, random_state=RANDOM_STATE))])
        g = GridSearchCV(pipe, {"svc__C": [0.1, 1, 10], "svc__gamma": [0.01, 0.1]},
                         cv=CV, scoring="roc_auc")
    g.fit(Xtr, ytr)
    return float(roc_auc_score(yte, g.predict_proba(Xte)[:, 1]))


leak_ok = False
if LEAK_PATH.exists() and not FORCE_RETRAIN:
    try:
        leak_res = json.loads(LEAK_PATH.read_text())
        leak_ok = leak_res.get("data_hash") == DATA_HASH
    except Exception:
        leak_ok = False

if leak_ok:
    print("[SKIP] leakage_demo: checkpoint cargado")
else:
    X_num = pd.get_dummies(X, drop_first=True).astype(float)
    np.random.seed(0)
    X_leaky = X_num.copy()
    X_leaky["leaky_feature"] = y.values + np.random.normal(0, 0.01, size=len(y))
    leak_res = {
        "data_hash": DATA_HASH,
        "sin_leaky_feature__escalar_antes_del_split": auc_flow(X_num, y, True),
        "sin_leaky_feature__pipeline": auc_flow(X_num, y, False),
        "con_leaky_feature__escalar_antes_del_split": auc_flow(X_leaky, y, True),
        "con_leaky_feature__pipeline": auc_flow(X_leaky, y, False),
    }
    LEAK_PATH.write_text(json.dumps(leak_res, indent=2))   # guardado inmediato
    print("[SAVED] leakage_demo")

leak_df = pd.Series({k: v for k, v in leak_res.items() if k != "data_hash"},
                    name="AUC test").to_frame()
display(leak_df.round(4))

[SAVED] leakage_demo


,AUC test
sin_leaky_feature__escalar_antes_del_split,0.9298
sin_leaky_feature__pipeline,0.9115
con_leaky_feature__escalar_antes_del_split,1.0000
con_leaky_feature__pipeline,1.0000


### Lectura de los resultados

| Escenario | AUC en prueba |
|---|---|
| Sin `leaky_feature`, escalado **antes** del split (con fuga de preprocesamiento) | 0.9298 |
| Sin `leaky_feature`, `Pipeline` (flujo correcto) | 0.9115 |
| Con `leaky_feature`, escalado antes del split | 1.0000 |
| Con `leaky_feature`, `Pipeline` | 1.0000 |

* **Con `leaky_feature` el AUC es 1.0 en ambos flujos.** Esa variable se construyó a partir del
  objetivo (`y` más un ruido de 0.01), así que contiene la respuesta. El `Pipeline` no puede
  evitar esta fuga: solo controla qué datos usan el imputador y el escalador, no qué variables
  se le entregan al modelo. Es la fuga del **objetivo**, y se previene revisando de dónde sale
  cada variable.
* **Sin esa variable**, escalar antes del split dio un AUC mayor (0.9298) que el `Pipeline`
  (0.9115). La dirección es la esperada (el flujo con fuga es más optimista), pero la diferencia
  de 0.018 sale de una sola partición con 184 pacientes de prueba, así que no sirve para medir
  *cuánto* infla la fuga. Para eso habría que repetir el experimento con muchas particiones.
* Lo que justifica el `Pipeline` es el **procedimiento**: el imputador y el escalador se ajustan
  solo con los datos de entrenamiento de cada pliegue, y los datos de prueba nunca influyen en
  ninguna estadística.

### Datos y preparación

* 918 pacientes, 11 variables predictoras y el objetivo `HeartDisease` (55.3 % con enfermedad,
  44.7 % sin ella).
* No hay valores nulos explícitos, pero hay valores imposibles: **172 colesteroles = 0** y
  **1 presión en reposo = 0**. Se convirtieron a `NaN` y se imputan con la mediana **dentro del
  `Pipeline`** (la mediana se calcula solo con los datos de entrenamiento).
* Partición 80/20 estratificada con `random_state=42`: 734 pacientes para entrenamiento y 184
  para prueba, hecha antes de escalar o imputar.